In [ ]:
##################   OHLC DOWNLOADER + DOWNLOADER FROM LAST SUCCESFUL TICKER ############XX 
nasdaq_traded_df = pd.read_csv('nasdaqtraded.txt', sep='|')
other_listed_df = pd.read_csv('otherlisted.txt', sep='|')

# Combine both dataframes into one
combined_df = pd.concat([nasdaq_traded_df, other_listed_df])

# Select the ticker column and drop duplicates
tickers = combined_df['Symbol'].dropna().unique()  # Handle NaN values

# Define output directory
output_dir = "C:/Users/j-vas/1JUPYTERLAB/diplomka/OHLC_all"

# Ensure output directory exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Function to download OHLC data
def download_ohlc(ticker):
    try:
        stock = yf.Ticker(ticker)
        final = stock.history(period="max")

        if not final.empty:
            final.to_csv(f'{output_dir}/{ticker}.csv')
            print(f"Downloaded data for {ticker}")
        else:
            raise ValueError("No data available")

    except Exception as e:
        # Handle delisting or suffix issues
        error_message = str(e)
        print(f"Got error for {ticker}: {error_message}")
        
        # Retry with stripped suffixes ('.' and '$')
        stripped_ticker = ticker.split('.')[0].split('$')[0]
        if stripped_ticker != ticker:
            print(f"Retrying download for {stripped_ticker} after removing suffix...")
            try:
                stock = yf.Ticker(stripped_ticker)
                final = stock.history(period="max")

                if not final.empty:
                    final.to_csv(f'{output_dir}/{stripped_ticker}.csv')
                    print(f"Downloaded data for {stripped_ticker}")
                else:
                    raise ValueError("No data available")
            except Exception as e:
                print(f"Failed to download {stripped_ticker}: {e}")

# Function to continue from the last ticker
def download_from_last(last_downloaded):
    # Find the index of the last downloaded ticker
    start_index = list(tickers).index(last_downloaded) + 1 if last_downloaded in tickers else 0
    for ticker in tickers[start_index:]:
        if isinstance(ticker, str):  # Ensure the ticker is a string
            download_ohlc(ticker)  # Call the main download function
        time.sleep(3)

# Start downloading from 'AAAAAA'
last_successful_ticker = 'AAAAAA'
download_from_last(last_successful_ticker)


In [ ]:
################ DELETING INACTIVE STOCKS (LAST 3 YEARS) FROM DIRECTORY ################### DONT RUN MORE THAN ONCE
'''
directory = "C:/Users/j-vas/1JUPYTERLAB/diplomka/OHLC_all"
threshold = 30000
csv_files = glob.glob(os.path.join(directory, "*.csv"))

df_list = []
date_threshold = pd.Timestamp('2022-01-01')

# To track tickers/files to delete
files_to_delete = []

for file in csv_files:
    df = pd.read_csv(file)
    
    # Split Date into separate Date and Time columns
    df['Date'] = df['Date'].astype(str)  # Ensure it's a string if it's not already
    df['Time'] = df['Date'].str.split().str[1]  # Extract the time part
    df['Date'] = df['Date'].str.split().str[0]  # Extract the date part
    
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
    
    # Filter for data after the date threshold
    recent_data = df[df['Date'] >= date_threshold].copy()
    
    if not recent_data.empty:
        # Calculate usd_volume and rolling mean
        recent_data['usd_volume'] = recent_data['Close'] * recent_data['Volume']
        recent_data['rolling_mean_usd_volume'] = recent_data['usd_volume'].rolling(40).mean()
        
        # Check the rolling mean for the last 3 years
        if recent_data['rolling_mean_usd_volume'].min() > threshold:
            recent_data['ticker'] = os.path.basename(file).replace('.csv', '')  # Add 'ticker' column
            df_list.append(recent_data)
        else:
            # Mark file for deletion if rolling mean is below threshold
            files_to_delete.append(file)

# Concatenate and display tickers to keep
if df_list:
    filtered_stocks = pd.concat(df_list, ignore_index=True)
    tickers_to_keep = filtered_stocks['ticker'].unique()
else:
    print("No stocks meet the criteria.")

# Delete files for inactive stocks
for file in files_to_delete:
    os.remove(file)
    print(f"Deleted inactive stock file: {file}")
    
    
    
'''

In [4]:
############ GAP UP TRADES ################

gap_up_trades_list = []

for ticker in recent['Ticker'].unique():
    ticker_df = recent[recent['Ticker'] == ticker].sort_values(by='Date').reset_index(drop=True)

    for i in range(1, len(ticker_df)):
        # Check if the current day and the previous day are consecutive
        if (ticker_df.loc[i, 'Date'] - ticker_df.loc[i - 1, 'Date']).days == 1:
            prev_close = ticker_df.loc[i - 1, 'Close']  # Closing price at t-1
            curr_open = ticker_df.loc[i, 'Open']        # Opening price at t

            if curr_open >= 1.06 * prev_close:
                gap_up_trades_list.append(ticker_df.loc[i])  # Store the gap-up day

if gap_up_trades_list:
    gap_up_trades = pd.concat(gap_up_trades_list, axis=1).T

gap_up_trades.reset_index(drop=True, inplace=True)
gap_up_trades


,Ticker,Date,Open,High,Low,Close,Volume,usd_volume,rolling_mean_usd_volume
0,A,2017-05-23,56.4073,56.577089,54.709422,55.331978,7510600.0,415576352.796936,98032282.737892
1,A,2018-02-15,71.148921,71.196387,67.617108,68.376633,4899100.0,334983961.213684,138911720.898346
2,A,2019-08-15,69.226744,69.909966,66.801784,66.96537,6701000.0,448734945.56427,169133639.689461
3,A,2020-11-24,115.704758,115.704758,108.380811,111.541878,5559900.0,620161686.083222,142389025.548065
4,A,2022-08-17,146.276421,146.433669,137.618164,139.839233,4110100.0,574753233.190918,182242360.592209
...,...,...,...,...,...,...,...,...,...
62405,ZYXI,2020-06-11,20.733727,22.828678,20.302149,20.54491,4183960.0,85959083.446503,16628637.363833
62406,ZYXI,2020-06-12,21.803682,22.001488,18.665748,19.951492,2664860.0,53167933.796082,17623760.127401
62407,ZYXI,2020-07-29,18.22518,18.243164,16.076281,16.480885,2349490.0,38721673.446083,29568471.631497
62408,ZYXI,2022-05-03,6.56,6.83,6.14,6.52,440300.0,2870755.991602,2668762.946615


In [ ]:
#########################    1-MIN OHLC to folder with 60k trades FROM GAPUPTRADES  -- SYMBOLS A  and csv with failed_downloads   ############
# Define paths
ohlc_data_dir = "C:/Users/j-vas/1JUPYTERLAB/diplomka/1MIN STOCK DATA/stock_Z_full_1min"
output_dir = "C:/Users/j-vas/1JUPYTERLAB/diplomka/gap_trades/gap_trades_Z"
failed_downloads = []  # List to track failed downloads

# Ensure 'Date' column is in datetime format
gap_up_trades['Date'] = pd.to_datetime(gap_up_trades['Date'])
gap_up_trades_A = gap_up_trades[gap_up_trades['Ticker'].str.startswith('Z')]

# Process each unique ticker in gap-up trades
for ticker in gap_up_trades_A['Ticker'].unique():
    # Find all dates for the current ticker
    ticker_dates = gap_up_trades_A[gap_up_trades_A['Ticker'] == ticker]['Date'].dt.strftime('%Y-%m-%d').tolist()

    # Construct the file path for the ticker's OHLC data
    ticker_file = os.path.join(ohlc_data_dir, f"{ticker}_full_1min_adjsplitdiv.txt")
    print(f"Looking for file: {ticker_file}")  # Debugging output

    # Check if the ticker file exists
    if not os.path.exists(ticker_file):
        print(f"OHLC data for {ticker} not found. Skipping...")
        failed_downloads.append({'Ticker': ticker, 'Reason': 'File not found'})
        continue

    # Read the OHLC data (without column names)
    try:
        ohlc_data = pd.read_csv(
            ticker_file, 
            header=None,  # No headers in the file
        )
    except Exception as e:
        print(f"Error reading {ticker_file}: {e}")
        failed_downloads.append({'Ticker': ticker, 'Reason': str(e)})
        continue

    # Rename columns for internal use
    ohlc_data.columns = ['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']

    # Extract the date and time part from the Datetime column
    ohlc_data['Datetime'] = pd.to_datetime(ohlc_data['Datetime'])
    ohlc_data['Date'] = ohlc_data['Datetime'].dt.strftime('%Y-%m-%d')
    ohlc_data['Time'] = ohlc_data['Datetime'].dt.strftime('%H:%M')

    # Filter rows between 4:00 AM and 10:30 AM
    ohlc_data_filtered = ohlc_data[(ohlc_data['Time'] >= '04:00') & (ohlc_data['Time'] <= '10:30')]

    for date in ticker_dates:
        # Filter for the rows corresponding to the specific date
        filtered_data = ohlc_data_filtered[ohlc_data_filtered['Date'] == date]

        if filtered_data.empty:
            print(f"No data for {ticker} on {date}. Skipping...")
            failed_downloads.append({'Ticker': ticker, 'Date': date, 'Reason': 'No data for date'})
            continue

        # Add Ticker and Gap-Up Date columns
        filtered_data = filtered_data.copy()  # Avoid SettingWithCopyWarning
        filtered_data.loc[:, 'Ticker'] = ticker  # Use .loc to avoid the warning
        filtered_data.loc[:, 'Date'] = date  # Use .loc to avoid the warning

        # Save filtered data to a separate CSV for this ticker and date
        output_file = os.path.join(output_dir, f"{ticker}_{date}.csv")
        filtered_data.to_csv(output_file, index=False)
        print(f"Data saved for {ticker} on {date} to {output_file}")

# After processing all trades, save the failed downloads to a CSV
if failed_downloads:
    failed_downloads_df = pd.DataFrame(failed_downloads)
    failed_downloads_df.to_csv(os.path.join(output_dir, 'failed_downloads.csv'), index=False)
    print(f"Failed downloads saved to {output_dir}/failed_downloads.csv")
else:
    print("No failed downloads.")
